In [2]:
import os, httpx
from dotenv import load_dotenv


load_dotenv()

True

In [3]:
import requests

# Build client so connections can be re-used across
# requests. This only needs to happen once per session.
client = requests.Session()

resp = client.post(
    "https://inference.baseten.co/v1/systemone",
    headers={"Authorization": f"Api-Key {os.environ['MAMMOTH_API_KEY']}"},
    json={
        "state": "I was charged twice for my subscription. Please refund the duplicate charge.",
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which team should handle this request?",
                "criteria": {
                    "billing": "Billing, payments, and refunds",
                    "technical_support": "Technical troubleshooting"
                }
            },
            "refund": {
                "type": "noul",
                "instructions": "Is the customer asking for a refund?"
            },
            "urgency": {
                "type": "score",
                "instructions": "How urgent is the request?",
                "criteria": [
                    "Routine request",
                    "Needs attention today",
                    "Immediate emergency"
                ]
            }
        },
        "model": "decision-models/baseten-mammoth-27b-latest"
    },
)

print(resp.json())

{'model': 'decision-models/baseten-mammoth-27b-latest', 'answers': {'route': {'type': 'choice', 'choice': 'billing', 'confidence': 0.9976393977219136, 'probabilities': {'billing': 0.9988196988609568, 'technical_support': 0.0011803011390431925}}, 'refund': {'type': 'noul', 'noul': 0.9993005831900714}, 'urgency': {'type': 'score', 'score': 0.8621165966816995, 'confidence': 0.34405353921263304, 'legend': {'0': 'Routine request', '1': 'Needs attention today', '2': 'Immediate emergency'}, 'probabilities': {'0': 0.28759052192160584, '1': 0.5627023594750886, '2': 0.1497071186033054}}}, 'usage': {'input_tokens': 231, 'output_tokens': 3}}


In [4]:
import time
from types import SimpleNamespace

MAMMOTH_URL = "https://inference.baseten.co/v1/systemone"
MAMMOTH_MODEL = "decision-models/baseten-mammoth-27b-latest"
TIMEOUT = httpx.Timeout(600.0, connect=30.0)

# Regeln einmal im State statt als Criteria pro Kandidat (wie bei Jev/Drex)
RULES = ("A column counts as referenced if it appears in SELECT, WHERE, a JOIN condition, GROUP BY, "
         "HAVING, ORDER BY, or inside an aggregation function of the SQL query that answers the question. "
         "If a table is used only via * or COUNT(*) and no other column of it is referenced, "
         "its first column counts as referenced.")


# Rate Limits (Baseten): 100K Tokens/min, 800 Requests/min. Wir bleiben mit einem gleitenden 60-s-Fenster darunter.
import collections
import json as _json

TOKENS_PER_MIN = 90_000
REQUESTS_PER_MIN = 720  # Limit laut Baseten: 800/min
_window = collections.deque()  # (zeitpunkt, tokens) der Requests der letzten 60 s


def _throttle(est_tokens):
    while True:
        now = time.monotonic()
        while _window and now - _window[0][0] > 60:
            _window.popleft()
        used = sum(t for _, t in _window)
        if len(_window) < REQUESTS_PER_MIN and used + est_tokens <= TOKENS_PER_MIN:
            _window.append((now, est_tokens))
            return
        time.sleep(max(0.5, 60 - (now - _window[0][0]) + 0.1))


def query_mammoth(state, questions, retries=8):
    """Ein Mammoth-Request. questions: {name: instruction}.
    Liefert ein Objekt mit .nouls[name].noul und .usage.input_tokens/.output_tokens
    (gleiche Form wie bei Jev/Drex, damit die Eval-Funktionen unveraendert funktionieren)."""
    payload = {
        "model": MAMMOTH_MODEL,
        "state": state,
        "questions": {name: {"type": "noul", "instructions": instr} for name, instr in questions.items()},
    }
    est_tokens = int(len(_json.dumps(payload)) / 2.0)  # grobe Schaetzung wie im Rest des Projekts (chars/2)
    for attempt in range(retries):
        _throttle(est_tokens)
        try:
            response = httpx.post(
                MAMMOTH_URL,
                headers={"Authorization": f"Api-Key {os.environ['MAMMOTH_API_KEY']}"},
                json=payload,
                timeout=TIMEOUT,
            )
            if response.status_code >= 400:
                # Body und Rate-Limit-Header mit ausgeben, damit man sieht, WELCHES Limit greift
                hdrs = {k: v for k, v in response.headers.items() if "limit" in k.lower() or "retry" in k.lower()}
                raise httpx.HTTPStatusError(f"{response.status_code}: {response.text[:300]} {hdrs}",
                                            request=response.request, response=response)
            break
        except (httpx.HTTPStatusError, httpx.TransportError) as e:
            status = getattr(getattr(e, "response", None), "status_code", None)
            retryable = status is None or status == 429 or status >= 500
            if attempt == retries - 1 or not retryable:
                raise
            wait = 2 ** attempt
            if status == 429:
                # Limits gelten pro Minute -> Retry-After beachten, sonst mind. 20 s warten
                wait = max(float(e.response.headers.get("retry-after", 0) or 0), 20)
            time.sleep(wait)
    body = response.json()
    return SimpleNamespace(
        nouls={k: SimpleNamespace(noul=a["noul"]) for k, a in body["answers"].items()},
        usage=SimpleNamespace(input_tokens=body["usage"]["input_tokens"],
                              output_tokens=body["usage"]["output_tokens"]),
    )


def merge_responses(responses):
    # Fuehrt die Responses aller Chunks einer Frage zu einem Objekt mit .nouls und .usage zusammen.
    return SimpleNamespace(
        nouls={k: a for resp in responses for k, a in resp.nouls.items()},
        usage=SimpleNamespace(
            input_tokens=sum(resp.usage.input_tokens for resp in responses),
            output_tokens=sum(resp.usage.output_tokens for resp in responses),
        ),
    )

In [5]:
# Zweistufiges Schema Linking mit Mammoth (analog zu jev_test.ipynb / drex_max_test.ipynb):
#   Stufe 1: kompaktes Schema (Tabellen, Spalten, FK-Ziele) -> ein Noul pro Tabelle
#   Stufe 2: volle DDL nur der ausgewaehlten Tabellen -> ein Noul pro Spalte
# Kontextfenster von Mammoth: 16k Tokens. Die Chunk-Groessen sind dieselben wie bei Jev (12000 Zeichen / 6000 Tokens),
# damit die Ergebnisse vergleichbar bleiben; dazu kommen noch die Fragen (ein Noul pro Kandidat).
import random

from src.utils import _extract_fk_targets, _pack_tables_into_chunks

TABLE_RULES = ("A table is needed if at least one of its columns is referenced in the SQL query that answers the question, "
               "or if it is needed to join other needed tables. "
               "The schema shown is only one part of a large database; the needed tables may be in another part.")


def compact_table(t):
    # 'tabelle(spalte1, spalte2) FK -> ziel1, ziel2' - ohne Typen, Backticks und Beispielwerte
    line = f"{t['candidates']['table']}({', '.join(t['candidates']['columns'])})"
    fks = _extract_fk_targets(t['ddl'])
    if fks:
        line += f" FK -> {', '.join(sorted(fks))}"
    return line


def chunk_compact(tables, max_chars):
    # Greedy wie _pack_tables_into_chunks, aber nach Zeichen auf dem kompakten Format
    chunks, current, size = [], [], 0
    for t in tables:
        line = compact_table(t)
        if current and size + len(line) > max_chars:
            chunks.append(current)
            current, size = [], 0
        current.append((t['candidates']['table'], line))
        size += len(line) + 1
    if current:
        chunks.append(current)
    return chunks


def fk_neighbors(tables, selected):
    # Tabellen, die per FK direkt mit einer ausgewaehlten Tabelle verbunden sind (beide Richtungen)
    neighbors = set()
    for t in tables:
        name = t['candidates']['table'].lower()
        targets = {x.lower() for x in _extract_fk_targets(t['ddl'])}
        if name in selected:
            neighbors |= targets
        if targets & selected:
            neighbors.add(name)
    return neighbors - selected


_consecutive_errors = 0


def _ask(q, state, questions, responses):
    # Fehler werden nur geloggt, aber bei 5 Fehlern in Folge brechen wir ab, statt den Rest des Laufs
    # mit lauter fehlgeschlagenen Requests zu verbrennen (und ein unvollstaendiges Ergebnis zu speichern).
    global _consecutive_errors
    try:
        responses.append(query_mammoth(state=state, questions=questions))
        _consecutive_errors = 0
    except Exception as e:
        _consecutive_errors += 1
        print(f"Error processing question {q['question']} with db_id {q['db']}: {e}")
        if _consecutive_errors >= 5:
            raise RuntimeError("5 Requests in Folge fehlgeschlagen - Lauf abgebrochen") from e


def run_mammoth_two_stage(data, table_threshold=0.3, include_fk_neighbors=False,
                          stage1_max_chars=12000, stage2_max_tokens=6000):
    results = []
    for q in data:
        t0 = time.perf_counter()

        # Stufe 1: Tabellen
        table_responses = []
        stage1_chunks = chunk_compact(q['db_schema'], stage1_max_chars)
        for chunk in stage1_chunks:
            _ask(q,
                 state={"question": q['question'], "schema": "\n".join(line for _, line in chunk), "rules": TABLE_RULES},
                 questions={name: f"Is table {name} needed for the SQL query?" for name, _ in chunk},
                 responses=table_responses)
        table_scores = {k: a.noul for resp in table_responses for k, a in resp.nouls.items()}

        selected = {t.lower() for t, p in table_scores.items() if p >= table_threshold}
        if include_fk_neighbors:
            selected |= fk_neighbors(q['db_schema'], selected)
        selected_tables = [t for t in q['db_schema'] if t['candidates']['table'].lower() in selected]

        # Stufe 2: Spalten der ausgewaehlten Tabellen
        column_responses = []
        stage2_chunks = _pack_tables_into_chunks(selected_tables, q['question'], stage2_max_tokens, q['db'])
        for chunk in stage2_chunks:
            _ask(q,
                 state={"question": q['question'], "schema": chunk['ddl'], "rules": RULES},
                 questions={c: f"Is column {c} referenced in the SQL query?" for c in chunk['candidate_strings']},
                 responses=column_responses)

        results.append({"question": q['question'], "db_id": q['db'], "sql": q['sql'], "gold_schema": q['gold_schema'],
                        "table_scores": table_scores, "selected_tables": sorted(selected),
                        "n_chunks_stage1": len(stage1_chunks), "n_chunks_stage2": len(stage2_chunks),
                        "seconds": time.perf_counter() - t0,
                        "table_response": merge_responses(table_responses),
                        "response": merge_responses(column_responses)})
    return results


def stage1_table_recall(results, thresholds=(0.1, 0.2, 0.3, 0.4, 0.5)):
    # Recall auf Tabellenebene: Wie viele Gold-Tabellen haette Stufe 1 bei welcher Schwelle durchgelassen?
    print(f"{'thr':>5} {'R_tab':>7} {'R=1':>7} {'#tab':>7}")
    for thr in thresholds:
        hit = total = n_sel = all_found = 0
        for r in results:
            gold = {t.lower() for t in r['gold_schema'] if t}
            sel = {t.lower() for t, p in r['table_scores'].items() if p >= thr}
            hit += len(gold & sel)
            total += len(gold)
            n_sel += len(sel)
            all_found += gold <= sel
        print(f"{thr:>5.2f} {hit / total:>7.4f} {all_found / len(results):>7.4f} {n_sel / len(results):>7.2f}")


def print_timing(results):
    n = len(results)
    c1 = sum(r['n_chunks_stage1'] for r in results) / n
    c2 = sum(r['n_chunks_stage2'] for r in results) / n
    secs = sorted(r['seconds'] for r in results)
    print(f"Chunks/Frage: Stufe 1 = {c1:.2f}, Stufe 2 = {c2:.2f}")
    print(f"Sekunden/Frage: Mittel = {sum(secs) / n:.2f}, Median = {secs[n // 2]:.2f}, max = {secs[-1]:.2f}")

In [6]:
# Eval: micro P/R/F6 ueber (table, column)-Paare, identisch zu jev_test.ipynb / drex_max_test.ipynb
import json


def response_to_dict(resp):
    return {
        "nouls": {k: a.noul for k, a in resp.nouls.items()},
        "usage": {"input_tokens": resp.usage.input_tokens,
                  "output_tokens": resp.usage.output_tokens},
    }


def save_results(results, file_name):
    serializable = []
    for r in results:
        item = {**r, "response": response_to_dict(r["response"])}
        if "table_response" in r:
            item["table_response"] = response_to_dict(r["table_response"])
        serializable.append(item)
    with open(f"{file_name}.json", "w", encoding="utf-8") as f:
        json.dump(serializable, f, indent=4)


def _gold_cols(gold):
    return {(t.lower(), c.lower()) for t, cols in gold.items() for c in (cols or []) if t and c}

def _scores(response):
    # 'table.column' -> noul-Wahrscheinlichkeit
    return {tuple(k.lower().split(".", 1)): a.noul for k, a in response.nouls.items()}

def eval_results(results, threshold=0.5, beta=6):
    tp = n_pred = n_gold = 0
    perfect_recall = 0
    for r in results:
        gold = _gold_cols(r["gold_schema"])
        pred = {k for k, p in _scores(r["response"]).items() if p >= threshold}
        tp += len(pred & gold)
        n_pred += len(pred)
        n_gold += len(gold)
        perfect_recall += gold <= pred
    p = tp / n_pred if n_pred else 0.0
    rec = tp / n_gold if n_gold else 0.0
    b2 = beta ** 2
    f = (1 + b2) * p * rec / (b2 * p + rec) if (b2 * p + rec) else 0.0
    return {"threshold": threshold, "precision": p, "recall": rec, f"f{beta}": f,
            "perfect_recall_rate": perfect_recall / len(results),
            "avg_pred_cols": n_pred / len(results), "tp": tp, "pred": n_pred, "gold": n_gold}

def present_results(results):
    print(f"Fragen: {len(results)}")

    # Gold-Spalten, die gar nicht unter den Kandidaten sind
    missing = sum(len(_gold_cols(r["gold_schema"]) - set(_scores(r["response"]))) for r in results)
    print(f"Gold-Spalten ohne Kandidat: {missing}")

    usage_in = sum(r["response"].usage.input_tokens for r in results)
    usage_out = sum(r["response"].usage.output_tokens for r in results)
    print(f"Tokens: input={usage_in:,}  output={usage_out:,}\n")

    print(f"{'thr':>5} {'P':>7} {'R':>7} {'F6':>7} {'R=1':>7} {'#pred':>7}")
    for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
        m = eval_results(results, thr)
        print(f"{thr:>5.2f} {m['precision']:>7.4f} {m['recall']:>7.4f} {m['f6']:>7.4f} "
              f"{m['perfect_recall_rate']:>7.4f} {m['avg_pred_cols']:>7.2f}")

    # Fehleranalyse bei 0.5: Fragen mit verpassten Gold-Spalten
    misses = []
    for r in results:
        s = _scores(r["response"])
        missed = {k: s.get(k) for k in _gold_cols(r["gold_schema"]) if s.get(k, 0) < 0.5}
        if missed:
            misses.append((r["question"], r["sql"], missed))
    print(f"\nFragen mit verpassten Gold-Spalten @0.5: {len(misses)}")
    for qtext, sql, missed in misses[:10]:
        print(f"- {qtext}\n  {sql}\n  missed: {missed}")

In [7]:
from src.data_loaders.spider_ent_data import get_spider_ent_data

# Gleiche 30 Fragen wie im Jev- und Drex-Test (seed 42) -> direkt vergleichbar
spider_ent = get_spider_ent_data(max_tokens=6000)
sample = random.Random(42).sample(spider_ent, 30)

results_two_stage = run_mammoth_two_stage(sample, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

# 'Gold-Spalten ohne Kandidat' zaehlt hier die Gold-Spalten, deren Tabelle in Stufe 1 rausgefallen ist
print("Stufe 2 (Spalten):")
present_results(results_two_stage)
print()
print_timing(results_two_stage)

save_results(results_two_stage, "spider_ent_mammoth_two_stage_results_30_samples_seed_42")

Stufe 1 (Tabellen):
  thr   R_tab     R=1    #tab
 0.10  1.0000  1.0000   28.70
 0.20  1.0000  1.0000   20.80
 0.30  1.0000  1.0000   16.90
 0.40  1.0000  1.0000   14.10
 0.50  1.0000  1.0000   11.33
Tokens Stufe 1: input=1,584,404

Stufe 2 (Spalten):
Fragen: 30
Gold-Spalten ohne Kandidat: 0
Tokens: input=505,141  output=5,769

  thr       P       R      F6     R=1   #pred
 0.10  0.1195  1.0000  0.8340  1.0000   22.87
 0.20  0.1429  1.0000  0.8605  1.0000   19.13
 0.30  0.1633  0.9878  0.8692  0.9667   16.53
 0.40  0.1837  0.9878  0.8833  0.9667   14.70
 0.50  0.2005  0.9878  0.8930  0.9667   13.47
 0.60  0.2308  0.9878  0.9074  0.9667   11.70
 0.70  0.2523  0.9878  0.9157  0.9667   10.70
 0.80  0.3105  0.9390  0.8903  0.8667    8.27
 0.90  0.3842  0.8902  0.8596  0.7667    6.33

Fragen mit verpassten Gold-Spalten @0.5: 1
- What are the ids of the TV channels that do not have any cartoons directed by Ben Jones?
  SELECT id FROM TV_Channel EXCEPT SELECT channel FROM cartoon WHERE direct

In [ ]:
results_two_stage = run_mammoth_two_stage(spider_ent, table_threshold=0.3, include_fk_neighbors=False)

print("Stufe 1 (Tabellen):")
stage1_table_recall(results_two_stage)
print(f"Tokens Stufe 1: input={sum(r['table_response'].usage.input_tokens for r in results_two_stage):,}\n")

print("Stufe 2 (Spalten):")
present_results(results_two_stage)
print()
print_timing(results_two_stage)

save_results(results_two_stage, "spider_ent_mammoth_two_stage_results")